In [1]:
pip install torch torchvision

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import copy
import random

In [5]:
# 1️⃣ Simple Neural Network Model
class SimpleModel(nn.Module):
    def __init__(self):
        super(SimpleModel, self).__init__()
        self.fc = nn.Linear(10, 2)

    def forward(self, x):
        return self.fc(x)

In [4]:
# 2️⃣ Create Dummy Client Data
def create_client_data(num_samples):
    X = torch.randn(num_samples, 10)
    y = torch.randint(0, 2, (num_samples,))
    return X, y

In [6]:
# 3️⃣ Client Local Training
def local_training(model, data, epochs=2):
    model.train()
    optimizer = optim.SGD(model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()

    X, y = data

    for epoch in range(epochs):
        optimizer.zero_grad()
        output = model(X)
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()

    return model.state_dict()

In [7]:
# 4️⃣ Weighted Federated Averaging (Server Side)
def weighted_fedavg(global_model, client_models, client_data_sizes):
    total_data = sum(client_data_sizes)
    new_state_dict = copy.deepcopy(global_model.state_dict())

    for key in new_state_dict.keys():
        new_state_dict[key] = sum(
            client_models[i][key] * (client_data_sizes[i] / total_data)
            for i in range(len(client_models))
        )

    global_model.load_state_dict(new_state_dict)
    return global_model

In [8]:
# 5️⃣ Federated Learning Simulation
def federated_learning(num_clients=3, rounds=3):
    global_model = SimpleModel()

    # Create clients with different data sizes
    client_data_sizes = [100, 200, 300]
    client_data = [create_client_data(size) for size in client_data_sizes]

    for round_num in range(rounds):
        print(f"\n🔄 Round {round_num + 1}")

        client_models = []

        for i in range(num_clients):
            local_model = copy.deepcopy(global_model)

            updated_weights = local_training(local_model, client_data[i])
            client_models.append(updated_weights)

            print(f"Client {i+1} trained with {client_data_sizes[i]} samples")

        # Server Aggregation
        global_model = weighted_fedavg(global_model, client_models, client_data_sizes)

        print("✅ Server aggregated models using Weighted FedAvg")

    return global_model

In [10]:
# Run Simulation
if __name__ == "__main__":
    final_model = federated_learning()
    print("\n Federated Learning Completed Successfully!")


🔄 Round 1
Client 1 trained with 100 samples
Client 2 trained with 200 samples
Client 3 trained with 300 samples
✅ Server aggregated models using Weighted FedAvg

🔄 Round 2
Client 1 trained with 100 samples
Client 2 trained with 200 samples
Client 3 trained with 300 samples
✅ Server aggregated models using Weighted FedAvg

🔄 Round 3
Client 1 trained with 100 samples
Client 2 trained with 200 samples
Client 3 trained with 300 samples
✅ Server aggregated models using Weighted FedAvg

 Federated Learning Completed Successfully!
